# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, ensuring machine-readable, standardized, and FAIR metadata.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata as object attributes
print(f"Dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview
Review available record sets and field definitions using `@id` keys.

In [ ]:
# Explore available record sets and their fields, referencing by @id
print("Available Record Sets:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"- Record Set @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {rs.description if hasattr(rs, 'description') else ''}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} (name: {field.name}, dataType: {field.data_type})")
    print()
# Store all record set @ids for use in the next section
record_set_ids = [rs.id for rs in record_sets]


## 3. Data Extraction
Load data from each record set into a DataFrame for detailed exploration. All record set and field references use their `@id`.

In [ ]:
dataframes = {}
# Loop through each record set @id and extract the records as DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id} with {len(df)} records and columns:")
    print(df.columns.tolist())
    print()
# Choose the first record set for further demonstration
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Preview of {example_record_set_id} record set:")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalization, or grouping. Use `@id` values for fields.

In [ ]:
# Let's inspect numeric fields in the chosen record set
df = dataframes[example_record_set_id]
print("Numeric columns detected:")
numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()
print(numeric_columns)
# We'll choose the first available numeric field for demonstration
if numeric_columns:
    numeric_field_id = numeric_columns[0]
    print(f"Selected numeric field for EDA: {numeric_field_id}")
    # Set a threshold (median for demonstration)
    threshold = df[numeric_field_id].median()
    print(f"Filtering rows where {numeric_field_id} > {threshold}")
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
    display(filtered_df.head())
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"First five normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Attempt grouping by a categorical field if present
    possible_group_fields = df.select_dtypes(include=["object", "category"]).columns.tolist()
    group_field_id = None
    for col in possible_group_fields:
        if df[col].nunique() > 1 and df[col].nunique() < 10:
            group_field_id = col
            print(f"Grouping by field: {group_field_id}")
            break
    if group_field_id:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped means of {numeric_field_id} by {group_field_id}:")
        display(grouped)
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the effect of grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated full FAIR data exploration using `mlcroissant` and `pandas`.

- We loaded the Croissant-based dataset and explored all `@id`-referenced record sets and fields.
- Extracted dataframes allow ad-hoc processing and statistical exploration.
- Using field `@id` ensures reproducibility and disambiguation for future workflow automation.

Continue to downstream analysis such as feature engineering, modeling, or exporting results.